# Análisis exploratorio con pandas

**Laboratorio de Modelo Analítico para la Toma de Decisiones**
ISTEA · 2026-2C · División A

---

La clase pasada dejamos el dataset **usable**. Hoy lo hacemos **hablar**.

Son dos cosas distintas y conviene tenerlas separadas en la cabeza:

| | Limpiar | Explorar |
| --- | --- | --- |
| **Sabés qué buscás** | Sí: duplicados, nulos, centinelas, outliers | No |
| **Termina cuando** | La bitácora está completa | Podés decir tres cosas que antes no sabías |
| **El resultado es** | Un archivo | Una frase |

| Parte | Qué es | Cuándo |
| --- | --- | --- |
| **A** | Explorar ElecCommSphere, paso a paso | Lo hacemos juntos |
| **B** | Lo mismo sobre **tu** dataset | Segunda hora, en equipo |

> **La meta de hoy:** salir de la clase con **tres hallazgos escritos** sobre tu
> dataset. Un hallazgo es una oración con sujeto, número y consecuencia — no una
> tabla ni un gráfico.

---
# PARTE A · ElecCommSphere

## A0 · Arrancamos del dataset limpio

Todo lo de hoy corre sobre el resultado de la clase pasada. Repetimos la cadena de
limpieza en una sola celda para tener el mismo punto de partida: **25.369 filas**,
el 96,4 % del original.

In [7]:
import pandas as pd
import numpy as np

tx  = pd.read_csv("eleccommsphere_transacciones.csv", parse_dates=["fecha"])
cli = pd.read_csv("eleccommsphere_clientes.csv", parse_dates=["fecha_alta"])

# La limpieza de la clase pasada, en cuatro pasos
t = tx.drop_duplicates()                                    # duplicados exactos
t = t[t["id_cliente"].isin(cli["id_cliente"])]              # clientes huérfanos
t = t[t["monto"].notna() & (t["monto"] > 0)]                # montos nulos o ≤ 0
t = t[t["monto"] <= t["monto"].quantile(0.999)]             # outliers de monto

print(f"Originales : {len(tx):,}")
print(f"Limpias    : {len(t):,}   ({len(t)/len(tx):.1%})")

Originales : 26,310
Limpias    : 25,369   (96.4%)


## A1 · La primera pasada: cuatro preguntas

No son cuatro cuentas, son cuatro preguntas. Se contestan en cinco minutos y ordenan
todo lo que viene después.

In [8]:
# 1. Granularidad: ¿qué representa una fila?
print("Filas                :", f"{len(t):,}")
print("Transacciones únicas :", f"{t['id_transaccion'].nunique():,}")
print("Clientes únicos      :", f"{t['id_cliente'].nunique():,}")
print("Filas por cliente    :", round(len(t) / t['id_cliente'].nunique(), 1))

# 2. Período real
print("\nDesde :", t["fecha"].min().date())
print("Hasta :", t["fecha"].max().date())
print("Meses :", t["fecha"].dt.to_period("M").nunique())

# 3. Tipos
print("\n", t.dtypes.to_string())

Filas                : 25,369
Transacciones únicas : 25,369
Clientes únicos      : 3,283
Filas por cliente    : 7.7

Desde : 2025-01-06
Hasta : 2026-06-30
Meses : 18

 id_transaccion                      int64
id_cliente                          int64
fecha                      datetime64[ns]
canal                              object
categoria                          object
monto                             float64
unidades                            int64
descuento_pct                     float64
metodo_pago                        object
busquedas_sesion                    int64
busquedas_sin_resultado             int64
devuelto                            int64


La cuarta pregunta —**¿hay columnas que no entiendo?**— no la contesta el código.
Se contesta mirando la lista y anotando las dudas. Una columna que no se entiende es
una pregunta para quien conoce el negocio, no una columna para borrar.

## A2 · describe(), y el primer hallazgo

`describe()` devuelve ocho números por columna. El truco no es correrlo: es saber
cuál mirar primero.

**Antes de correr la celda, apostá:** ¿la media va a ser mayor o menor que la
mediana?

In [9]:
t["monto"].describe().round(0)

,monto
count,25369.0
mean,182811.0
std,222541.0
min,4191.0
25%,48293.0
50%,102244.0
75%,229960.0
max,2403171.0


In [10]:
media   = t["monto"].mean()
mediana = t["monto"].median()

print(f"Media   : $ {media:,.0f}")
print(f"Mediana : $ {mediana:,.0f}")
print(f"Razón   : {media/mediana:.2f}")

Media   : $ 182,811
Mediana : $ 102,244
Razón   : 1.79


> **Hallazgo 1.** La media está un 79 % por encima de la mediana. Eso no es ruido:
> significa que la facturación está concentrada en pocas operaciones grandes.
>
> Al gerente no se le dice ni una ni la otra: **se le dice la diferencia entre las
> dos**, porque ahí está la información. La media sola exagera al cliente típico; la
> mediana sola esconde que existen compras enormes.

## A3 · ¿Quién concentra? El Pareto que no era 80/20

«El 20 % de los clientes hace el 80 % de la facturación» se repite en todas las
reuniones. Verificarlo lleva tres líneas.

In [11]:
por_cliente = t.groupby("id_cliente")["monto"].sum().sort_values(ascending=False)
acumulado   = por_cliente.cumsum() / por_cliente.sum()

for q in [0.10, 0.20, 0.30]:
    k = int(len(por_cliente) * q)
    print(f"Top {q:.0%} de clientes ({k:>4}) → {acumulado.iloc[k-1]:.1%} de la facturación")

print(f"\nHace falta el {(acumulado <= 0.80).mean():.1%} de los clientes para llegar al 80 %.")

Top 10% de clientes ( 328) → 37.5% de la facturación
Top 20% de clientes ( 656) → 56.5% de la facturación
Top 30% de clientes ( 984) → 69.8% de la facturación

Hace falta el 40.1% de los clientes para llegar al 80 %.


> **Hallazgo 2.** El 20 % de los clientes hace el 56 % de la facturación, no el 80 %.
> Para llegar al 80 % hace falta el 40 % de los clientes.
>
> La concentración existe y es fuerte, pero **no es la que dice la regla**. Si hubiéramos
> dado por buena la frase de manual, la estrategia de retención habría apuntado a un
> grupo la mitad de grande del que hace falta.

## A4 · El cruce que no dio nada

Agrupamos por canal esperando encontrar diferencias. Spoiler: no hay.

In [12]:
por_canal = t.groupby("canal").agg(
    operaciones = ("monto",    "size"),
    facturacion = ("monto",    "sum"),
    ticket      = ("monto",    "median"),
    devolucion  = ("devuelto", "mean"),
)
por_canal["% facturación"] = (por_canal["facturacion"] / por_canal["facturacion"].sum() * 100).round(1)
por_canal["devolucion"]    = (por_canal["devolucion"] * 100).round(1)
por_canal[["operaciones", "% facturación", "ticket", "devolucion"]].round(0)

,operaciones,% facturación,ticket,devolucion
canal,,,,
App,8718,34.0,100878.0,7.0
Instagram,1941,8.0,107217.0,7.0
Web,11660,46.0,102315.0,7.0
WhatsApp,3050,12.0,102349.0,6.0


> **Hallazgo 3.** Los cuatro canales tienen un ticket mediano casi idéntico —entre
> $ 100.900 y $ 107.200— y una tasa de devolución de entre 6 % y 7 %. **El canal no
> explica nada.**

Esto **también se reporta**. Un informe que solo trae lo que confirmó la hipótesis es
un informe sesgado, y decir «probamos esto y no explica nada» tiene un valor concreto:
evita que el equipo siga invirtiendo en segmentar por una variable que no segmenta.

## A5 · El tiempo: las tres series juntas

La facturación sola no alcanza. Si baja, ¿es porque compra menos gente o porque cada
uno gasta menos? Son dos problemas distintos y se arreglan de maneras distintas.

In [13]:
mensual = (t.set_index("fecha")
             .resample("ME")
             .agg(
    facturacion = ("monto",      "sum"),
    activos     = ("id_cliente", "nunique"),
    ticket      = ("monto",      "median"),
))

vista = mensual.copy()
vista["facturacion"] = (vista["facturacion"] / 1e6).round(0)
vista["ticket"]      = (vista["ticket"] / 1000).round(0)
vista.columns = ["facturación (M$)", "activos", "ticket (k$)"]
vista

,facturación (M$),activos,ticket (k$)
fecha,,,
2025-01-31,24.0,103,92.0
2025-02-28,67.0,277,116.0
2025-03-31,122.0,461,97.0
2025-04-30,157.0,586,101.0
2025-05-31,263.0,873,102.0
2025-06-30,228.0,842,100.0
2025-07-31,268.0,963,101.0
2025-08-31,265.0,943,111.0
2025-09-30,279.0,1021,101.0


Mirá la columna del medio y la de la derecha al mismo tiempo. Los activos suben hasta
noviembre de 2025 y después bajan. El ticket mediano **no se mueve**: se queda
alrededor de los $ 102.000 en los dieciocho meses.

In [14]:
pico = mensual["activos"].idxmax()
ult  = mensual.index[-1]

print(f"Pico de activos : {mensual.loc[pico, 'activos']:,} en {pico:%b-%Y}")
print(f"Último mes      : {mensual.loc[ult,  'activos']:,} en {ult:%b-%Y}")
print(f"Variación       : {mensual.loc[ult,'activos']/mensual.loc[pico,'activos']-1:+.0%}")
print()
print(f"Ticket en el pico     : $ {mensual.loc[pico, 'ticket']:,.0f}")
print(f"Ticket en el último   : $ {mensual.loc[ult,  'ticket']:,.0f}")

Pico de activos : 1,368 en Nov-2025
Último mes      : 991 en Jun-2026
Variación       : -28%

Ticket en el pico     : $ 103,667
Ticket en el último   : $ 101,256


> **El hallazgo que ordena toda la cursada.** Los clientes activos cayeron un 28 %
> desde el pico de noviembre y el ticket mediano quedó igual.
>
> **El problema no es el precio: es que se van clientes.** En la clase 1, Retención
> trajo el síntoma —«se nos están yendo clientes»—. Recién ahora hay un número que lo
> sostiene, y que además descarta la explicación alternativa.

## A6 · La trampa: el período no empieza donde creés

In [15]:
mensual.head(5)[["activos"]]

,activos
fecha,
2025-01-31,103
2025-02-28,277
2025-03-31,461
2025-04-30,586
2025-05-31,873


Enero de 2025 tiene 103 clientes activos. **No es un mes flojo: es el lanzamiento.**

Si calculás «el crecimiento del período» incluyendo la rampa inicial, te da un
+345 % y no significa nada. Cualquier negocio que arranca de cero crece infinito el
primer mes.

In [16]:
primeros = mensual["facturacion"].head(3).mean()
ultimos  = mensual["facturacion"].tail(3).mean()
print(f"Comparando primeros y últimos tres meses : {ultimos/primeros-1:+.0%}  ← engañoso")

desde_pico = mensual.loc[pico:, "activos"]
print(f"Desde el pico hasta el final (activos)   : {desde_pico.iloc[-1]/desde_pico.iloc[0]-1:+.0%}  ← esto sí")

Comparando primeros y últimos tres meses : +345%  ← engañoso
Desde el pico hasta el final (activos)   : -28%  ← esto sí


**Las otras dos trampas del día**, que en este dataset no aparecen pero en el tuyo
pueden:

- **El promedio que esconde dos poblaciones.** Si adentro hay dos grupos distintos,
  el promedio cae justo en el medio, donde no hay nadie. Antes de promediar, mirá la
  distribución: si tiene dos picos, el promedio miente.
- **El último período cortado.** Si el archivo termina el 15 del mes, ese mes va a
  mostrar la mitad de la facturación y va a parecer una caída. Los datasets de Amazon
  terminan a mitad de septiembre de 2023: ese mes hay que sacarlo.

## A7 · Los tres hallazgos, escritos

Así se entregan. Sujeto, número y consecuencia, en una oración cada uno.

1. **La facturación está concentrada:** el 20 % de los clientes hace el 56 % del
   total, y hace falta el 40 % para llegar al 80 %. La retención tiene que apuntar a
   un grupo más grande del que sugiere la regla de manual.
2. **El canal no segmenta:** los cuatro canales tienen el mismo ticket mediano
   (~$ 102.000) y la misma tasa de devolución (6-7 %). No conviene armar estrategias
   diferenciadas por canal.
3. **La caída es de clientes, no de precio:** los activos bajaron 28 % desde el pico
   de noviembre de 2025 mientras el ticket mediano se mantuvo estable. El problema a
   atacar es la retención, no el pricing.

Fijate que ninguna de las tres dice «se observa una tendencia decreciente».

---
# PARTE B · Tu dataset

## B0 · Las tres líneas de siempre

Todo lo de la Parte A funciona sobre tu archivo cambiando estos nombres. Si tu
dataset no tiene alguna de las columnas —muchos no tienen una variable de fricción—
dejala en `None` y salteá esa parte.

In [ ]:
import pandas as pd
import numpy as np

# ── Cargá tu dataset limpio de la clase pasada ────────────────────────────
# mi = pd.read_csv("mi_dataset_limpio.csv", parse_dates=["fecha"])

# ── Los cuatro roles, con los nombres de TU archivo ───────────────────────
ID       = "id_cliente"    # el identificador que se repite
FECHA    = "fecha"         # la fecha
VALOR    = "monto"         # el valor sumable
FRICCION = None            # cancelación, devolución, reseña de 1 estrella… o None

# Mientras no cargues el tuyo, esto corre sobre ElecCommSphere
mi = t.copy()
print("Trabajando sobre:", mi.shape)

## B1 · La primera pasada

In [ ]:
print("Filas                :", f"{len(mi):,}")
print("Identificadores      :", f"{mi[ID].nunique():,}")
print("Filas por id         :", round(len(mi) / mi[ID].nunique(), 1))
print("\nDesde :", mi[FECHA].min().date())
print("Hasta :", mi[FECHA].max().date())
print("Meses :", mi[FECHA].dt.to_period("M").nunique())
print("\n", mi.dtypes.to_string())

**Anotá acá las columnas que no entendés.** No hace falta resolverlas hoy: hace falta
que estén escritas.

## B2 · Media contra mediana

In [ ]:
print(mi[VALOR].describe().round(2))
print(f"\nMedia   : {mi[VALOR].mean():,.2f}")
print(f"Mediana : {mi[VALOR].median():,.2f}")
print(f"Razón   : {mi[VALOR].mean()/mi[VALOR].median():.2f}")

> Si tu dataset mide en **puntaje** (Amazon, MovieLens), la media y la mediana van a
> estar muy cerca: un puntaje va de 1 a 5 y no tiene cola larga. Eso también es un
> hallazgo — y te dice que tu concentración hay que buscarla en la **frecuencia**, no
> en el monto.

## B3 · ¿Quién concentra?

In [ ]:
por_id    = mi.groupby(ID)[VALOR].sum().sort_values(ascending=False)
acumulado = por_id.cumsum() / por_id.sum()

for q in [0.10, 0.20, 0.30]:
    k = int(len(por_id) * q)
    print(f"Top {q:.0%} ({k:>6}) → {acumulado.iloc[k-1]:.1%} del total")

print(f"\nHace falta el {(acumulado <= 0.80).mean():.1%} para llegar al 80 %.")

## B4 · Un groupby a elección

Elegí **una** columna categórica de tu dataset y compará los grupos. Puede ser
categoría de producto, país, vendedor, segmento, medio de pago, lo que tengas.

Puede pasar —como nos pasó con el canal— que no haya ninguna diferencia. Anotalo
igual.

In [ ]:
GRUPO = "canal"   # ← cambiá por una columna categórica de tu dataset

resumen = mi.groupby(GRUPO).agg(
    operaciones = (VALOR, "size"),
    total       = (VALOR, "sum"),
    tipico      = (VALOR, "median"),
)
resumen["% del total"] = (resumen["total"] / resumen["total"].sum() * 100).round(1)
resumen.sort_values("total", ascending=False).round(2)

## B5 · La serie mensual

Las tres columnas juntas. **Mirá la forma antes de concluir nada:** ¿hay rampa
inicial? ¿el último mes está cortado? ¿hay estacionalidad?

In [ ]:
serie = (mi.set_index(FECHA)
           .resample("ME")
           .agg(
    volumen = (VALOR, "size"),
    total   = (VALOR, "sum"),
    activos = (ID,    "nunique"),
    tipico  = (VALOR, "median"),
))

print("Primeros meses:"); display(serie.head(4).round(2))
print("Últimos meses:");  display(serie.tail(6).round(2))

In [ ]:
# ¿El último mes está completo?
ultimo = mi[FECHA].max()
print("Última fecha del archivo:", ultimo.date(), "— día", ultimo.day)
print("Si no es fin de mes, ese mes va a parecer una caída que no existió.")

# Pico y caída de identificadores activos
pico_b = serie["activos"].idxmax()
print(f"\nPico de activos : {serie.loc[pico_b,'activos']:,} en {pico_b:%b-%Y}")
print(f"Último mes      : {serie['activos'].iloc[-1]:,}")
print(f"Variación desde el pico: {serie['activos'].iloc[-1]/serie.loc[pico_b,'activos']-1:+.0%}")

## B6 · Escribí los tres

Completá la celda de abajo. Una oración cada uno, con la fórmula
**quién + qué hace distinto + cuánto**.

La prueba: si alguien que conoce ese negocio lo escucha y contesta «eso ya lo sabía»,
todavía no es un hallazgo. Y de los tres, **al menos uno tiene que sorprenderlos a
ustedes**: si los tres confirman lo que ya suponían, miraron poco.

### Nuestros tres hallazgos

**Equipo:**
**Dataset:**

1.
2.
3.

---

**Lo que probamos y no dio nada:**

**Lo que nos gustaría mirar y este archivo no nos deja:**

---
## Cómo se entrega

Un archivo corto a Classroom antes de la próxima clase, con:

- las tres oraciones numeradas,
- debajo de cada una, **la línea de código que la produjo**.

No lleva nota. Es el material con el que trabajamos la clase que viene, cuando veamos
qué gráfico corresponde a cada tipo de hallazgo. Los que lleguen sin los tres escritos
van a estar mirando cómo lo hacen los demás.

## Los cuatro errores que hunden un EDA

1. **Correr `describe()` y seguir de largo.** Los ocho números no son el análisis: son
   el punto de partida.
2. **Tablas sin pregunta.** Cada `groupby` tiene que responder algo que te preguntaste
   antes de escribirlo.
3. **Concluir sobre una serie sin mirarle la forma.** La rampa inicial y el último mes
   cortado producen conclusiones falsas a partir de números correctos.
4. **Reportar solo lo que confirmó la hipótesis.** El cruce que no dio nada vale tanto
   como el que dio.

## Si se traban

Por mail, con las cuatro cosas de siempre: qué intentaron, qué esperaban, qué pasó, y
el mensaje de error completo copiado y pegado.